In [1]:
import numpy as np
import math
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time
import itertools

In [2]:
#define the shape of the environment (i.e., its states). States for each node are 0,1,2,...,m_star where

#half chain

n_half = 4
m_star = 3

def create_actions(n):
    n_site = 2*n-3
    d_array = np.linspace(1,(2)**(n_site),(2)**(n_site))
    d_array = d_array.astype(int)
    # decimal to base 2, since action can be 0 or 1
    power=np.ones((d_array.shape[0],1))*((2)**np.arange(n_site)[::-1])
    m_array = np.floor((d_array[:,None]%((2)*power))/power) #0 to n-2 indices request for links (whether or not) and n-1 to 2*n-4 (n-2 indices in total have swap requests)
    
    #removing all actions that have request and swap request at same node
    u=0
    for m in m_array:
        flag=0
        for k in range(n-1,len(m)):
            if m[k]==1:
                if m[k-(n-1)]==1 or m[k-(n-1)+1]==1:
                    flag=1
                    break
        if flag==1:
            m_array = np.delete(m_array,u,axis=0)
            u-=1
        u+=1

    actions = [None]*len(m_array)
    #converting to matrix form
    for x in range(len(m_array)):
        action = np.zeros([n,n])
        for i in range(n-1):
            action[i][i+1] = m_array[x][i]
            action[i+1][i] = m_array[x][i]
            if i>0 and i<n-1:
                action[i][i] = m_array[x][i+n-2]
        actions[x] = action
    return(actions)

all_actions_half = create_actions(n_half)
all_actions_half = np.unique(all_actions_half, axis=0)


from sympy.utilities.iterables import multiset_permutations

numbers = list(range(1, n_half))
result = []

def allowed_perms(item, n):
    allowed = True
    for i in range(n):
        if item[i]<=i and item[i]!=0:
            allowed = False
            break
    if allowed:
        for i in range(n):
            if item[i]>i+1:
                for j in range(i+1, item[i]):
                    if item[j] > item[i]:
                        allowed = False
                        break
    return allowed

for i in range(n_half):
    subsets = itertools.combinations(numbers, i)
    for subset in subsets:
        zero_count = n_half - len(subset)
        temp = list(subset) + [0]*zero_count
        temp_list = [None]
        
        temp_list = multiset_permutations(temp)
        
        if temp_list:
            for zero_added_perms in temp_list:
                if zero_added_perms:
                    if allowed_perms(zero_added_perms,n_half):
                        result.append(list(zero_added_perms))


def all_states_nodes(nodes, n):
    nodes = np.array(nodes)
    n_act_l = nodes[nodes>0]
    n_act_r = np.where(nodes>0)[0]
    n_a = len(n_act_l)

    d_array = np.linspace(1,(m_star+1)**n_a-1,(m_star+1)**n_a-1)
    d_array = d_array.astype(int)
    
    #converting state number in decimal to m_star base number produces the state 
    power=np.ones((d_array.shape[0],1))*((m_star+1)**np.arange(n_a)[::-1])
    m_array = np.floor((d_array[:,None]%((m_star+1)*power))/power) #m_array[i][j] has state of jth link of the ith possible state
    
    states = [None]*len(m_array)
    
    for x in range(len(m_array)):
    #writing the states in the matrix form
#         print(m_array[x])
        state = np.zeros([n,n])
        if n_a>0:
            for i in range(n_a):
                state[n_act_r[i]][n_act_l[i]] = m_array[x][i]
                state[n_act_l[i]][n_act_r[i]] = m_array[x][i]
#             print(state)
            states[x] = state
            
#     print("states")
#     print(states)
    return(np.unique(states,axis=0))

all_st = []
for k in result:
    if len(all_states_nodes(k, n_half))>0:
#         print(len(all_states_nodes(k)))
        all_st.append(all_states_nodes(k, n_half)) 
        
all_states_half=[]
for i in all_st:
    for j in i:
        all_states_half.append(j)

all_states_half.append(np.zeros([n_half,n_half]))
all_states_half = np.unique(all_states_half,axis=0) 
print(len(all_states_half))

print([len(all_states_half), len(all_actions_half)])

D1_s_half = []
D2_s_half = []
for state in all_states_half:
    D1_s_half.append(np.linalg.eig(state)[0][0])
    D2_s_half.append(np.linalg.eig(state)[0][1])  
    
D1_s_half=np.array(D1_s_half)
D2_s_half=np.array(D2_s_half)

D1_a_half = []
D2_a_half = []
for action in all_actions_half:
    D1_a_half.append(np.linalg.eig(action)[0][0])
    D2_a_half.append(np.linalg.eig(action)[0][1])  
    
D1_a_half=np.array(D1_a_half)
D2_a_half=np.array(D2_a_half)

n_state_half = len(all_states_half)
n_action_half = len(all_actions_half)

q_values_half = np.zeros([n_state_half,n_action_half])



#full chain
n = 7
m_star = 3
all_actions = create_actions(n)
all_actions = np.unique(all_actions, axis=0)

numbers = list(range(1, n))
result = []

def allowed_perms(item, n):
    allowed = True
    for i in range(n):
        if item[i]<=i and item[i]!=0:
            allowed = False
            break
    if allowed:
        for i in range(n):
            if item[i]>i+1:
                for j in range(i+1, item[i]):
                    if item[j] > item[i]:
                        allowed = False
                        break
    return allowed

for i in range(n):
    subsets = itertools.combinations(numbers, i)
    for subset in subsets:
        zero_count = n - len(subset)
        temp = list(subset) + [0]*zero_count
        temp_list = [None]
        
        temp_list = multiset_permutations(temp)
        
        if temp_list:
            for zero_added_perms in temp_list:
                if zero_added_perms:
                    if allowed_perms(zero_added_perms,n):
                        result.append(list(zero_added_perms))
                        

all_st = []
for k in result:
    if len(all_states_nodes(k, n))>0:
#         print(len(all_states_nodes(k)))
        all_st.append(all_states_nodes(k, n)) 
        
all_states=[]
for i in all_st:
    for j in i:
        all_states.append(j)

all_states.append(np.zeros([n,n]))
all_states = np.unique(all_states,axis=0) 
print(len(all_states))

print([len(all_states), len(all_actions)])

D1_s = []
D2_s = []
for state in all_states:
    D1_s.append(np.linalg.eig(state)[0][0])
    D2_s.append(np.linalg.eig(state)[0][1])  
    
D1_s=np.array(D1_s)
D2_s=np.array(D2_s)

D1_a = []
D2_a = []
for action in all_actions:
    D1_a.append(np.linalg.eig(action)[0][0])
    D2_a.append(np.linalg.eig(action)[0][1])  
    
D1_a=np.array(D1_a)
D2_a=np.array(D2_a)

n_state = len(all_states)
n_action = len(all_actions)

q_values = np.zeros([n_state,n_action])

100
[100, 13]
20071
[20071, 233]


In [3]:
# Train the Model
# Our next task is for our AI agent to learn about its environment by implementing a Q-learning model. 
# The learning process will follow these steps:
# Choose a random, non-terminal state for the agent to begin this new episode.
# Choose an action for the current state. Actions will be chosen using an epsilon greedy algorithm. 
# This algorithm will usually choose the most promising action for the AI agent, 
# but it will occasionally choose a less promising option in order to encourage the agent to explore the environment.
# Perform the chosen action, and transition to the next state (i.e., move to the next location).
# Receive the reward for moving to the new state, and calculate the temporal difference.
# Update the Q-value for the previous state and action pair.
# If the new (current) state is a terminal state, go to #1. Else, go to #2.
# This entire process will be repeated across 1000 episodes. 
# This will provide the AI agent sufficient opportunity to learn the shortest paths 
# Define Helper Functions

#full chain functions

def is_allowed_state(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s))[0]
    while not res and i<len(subset_states):
        res = np.array_equal(state, all_states[subset_states[i]])
        i+=1 
    return res
#     res = [np.array_equal(state, i) for i in all_states]
#     if  np.argwhere(np.array(res)==True).size==0:
#         return False
#     else:
#         return True

#define a function that determines if the specified location is a terminal state
def is_terminal_state(current_state):
  #if the state has a link between 1st and last node then it is terminal
    if current_state[0][n-1] > 0 and is_allowed_state(current_state):  
        return True
    else:
        return False

def get_state_index(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s))[0]
    while not res:
        res = np.array_equal(state, all_states[subset_states[i]])
        i+=1 
    return subset_states[i-1]


def get_action_index(action):
    i=0
    res = False
    subset_actions = np.nonzero(np.isclose(np.linalg.eig(action)[0][0],D1_a))[0]
    while not res:
        res = np.array_equal(action, all_actions[subset_actions[i]])
        i+=1 
    return subset_actions[i-1]

def get_random_state():
    i = np.random.randint(0,len(all_states))
    return all_states[i]

def get_random_action():
    i = np.random.randint(0,len(all_actions))
    return all_actions[i]

#define a function that will choose a random, non-terminal starting state
def get_starting_state():
    
#   get a random state
    current_state = get_random_state()

#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state(current_state):
        current_state = get_random_state()

    return current_state


#define an epsilon greedy algorithm that will choose which action to take next
def get_next_action(current_state, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon, 
  #then choose the most promising value from the Q-table for this state.
    state_index = get_state_index(current_state)
    
    if np.random.random() < epsilon: #return opt_action 
        action_index = np.argmax(q_values[state_index])
        opt_action = all_actions[action_index]
        return opt_action
    else: #choose a random action
        return get_random_action()

#define a function that will get the next location based on the chosen action
def get_next_state(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n,n])
    
    for i in range(n):
        for j in range(n):
            new_state[i][j] = current_state[i][j]
    
    flag=0
    for i in range(n-1):
        if action[i][i+1]>0:
            flag=1
            break
            
    if flag==1:
        # wait                

        for i in range(n-1):
            for j in range(i+1,n):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                new_state[j][i] = new_state[i][j] 
            
    for i in range(n-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        
        
        for k in range(i+2,n):
            if new_state[i][k]>0:
                new_state[i][k] = 0
        
        for k in range(0,i):
            if new_state[i+1][k]>0:
                new_state[i+1][k] = 0

        new_state[j][i] = new_state[i][j]   
    
    
    #Bell measurements
    bell_flag = 0
    for i in range(1,n-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = max(node1_val,node2_val)
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n):
            for j in range(i+1,n):
                new_state[j][i] = new_state[i][j]
            
    return new_state

In [4]:
#half chain functions

def is_allowed_state_half(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s_half))[0]
    while not res and i<len(subset_states):
        res = np.array_equal(state, all_states_half[subset_states[i]])
        i+=1 
    return res
#     res = [np.array_equal(state, i) for i in all_states_half]
#     if  np.argwhere(np.array(res)==True).size==0:
#         return False
#     else:
#         return True

#define a function that determines if the specified location is a terminal state
def is_terminal_state_half(state):
  #if the state has a link between 1st and last node then it is terminal
    if state[0][n_half-1]>0 and is_allowed_state_half(state):  
        return True
    else:
        return False

def get_state_index_half(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s_half))[0]
    while not res:
        res = np.array_equal(state, all_states_half[subset_states[i]])
        i+=1 
    return subset_states[i-1]


def get_action_index_half(action):
    i=0
    res = False
    subset_actions = np.nonzero(np.isclose(np.linalg.eig(action)[0][0],D1_a_half))[0]
    while not res:
        res = np.array_equal(action, all_actions_half[subset_actions[i]])
        i+=1 
    return subset_actions[i-1]

def get_random_state_half():
    i = np.random.randint(0,len(all_states_half))
    return all_states_half[i]

def get_random_action_half():
    i = np.random.randint(0,len(all_actions_half))
    return all_actions_half[i]

#define a function that will choose a random, non-terminal starting state
def get_starting_state_half():
    
#   get a random state
    current_state = get_random_state_half()

#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state_half(current_state):
        current_state = get_random_state_half()

    return current_state


#define an epsilon greedy algorithm that will choose which action to take next
def get_next_action_half(current_state, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon, 
  #then choose the most promising value from the Q-table for this state.
    state_index = get_state_index_half(current_state)
    
    if np.random.random() < epsilon: #return opt_action 
        action_index = np.argmax(q_values[state_index])
        opt_action = all_actions_half[action_index]
        return opt_action
    else: #choose a random action
        return get_random_action_half()

#define a function that will get the next location based on the chosen action
def get_next_state_half(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n_half,n_half])
    
    for i in range(n_half):
        for j in range(n_half):
            new_state[i][j] = current_state[i][j]
    
    flag=0
    for i in range(n_half-1):
        if action[i][i+1]>0:
            flag=1
            break
            
    if flag==1:
        # wait                

        for i in range(n_half-1):
            for j in range(i+1,n_half):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                new_state[j][i] = new_state[i][j] 
            
    for i in range(n_half-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        
        
        for k in range(i+2,n_half):
            if new_state[i][k]>0:
                new_state[i][k] = 0
        
        for k in range(0,i):
            if new_state[i+1][k]>0:
                new_state[i+1][k] = 0

        new_state[j][i] = new_state[i][j]   
    
    
    #Bell measurements
    bell_flag = 0
    for i in range(1,n_half-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n_half):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = max(node1_val,node2_val)
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n_half):
            for j in range(i+1,n_half):
                new_state[j][i] = new_state[i][j]
            
    return new_state

In [5]:
# Reward function
def rewards(state, D1_s, all_states):
    if is_terminal_state(state):
        reward = 100#/state[0][n-1]
    else:
        reward = -1            
    return reward

# Reward function half chain
def rewards_half(state):
    if is_terminal_state_half(state):
        reward = 100#/state[0][n-1]
    else:
        reward = -1            
    return reward

In [6]:
def training_network_half(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes):
    
    #run through 500 training episodes
    for episode in range(int(total_episodes)):
        if episode%1000==0:
            print(episode)
        
        #get the starting state for this episode
        state = get_starting_state_half()
#         print(state)
        
        #continue taking actions until we reach a terminal state or we do 1000 actions
        trial_num = 0
        while trial_num<200 and is_terminal_state_half(state)==False:
            #choose which action to take
            action = get_next_action_half(state, epsilon)
#             print(action)
#             print(" ")
            #perform the chosen action, and transition to the next state
            old_state = np.zeros([n_half,n_half]) #store the old state
            for i in range(n_half):
                for j in range(n_half):
                    old_state[i][j] = state[i][j]
                    
            state = get_next_state_half(old_state, action, p_l, p_bm)
#             print(state)
#             print(" ")
            old_state_index = get_state_index_half(old_state)
            state_index = get_state_index_half(state)
            action_index = get_action_index_half(action)
            
            #receive the reward for moving to the new state, and calculate the temporal difference
            reward = rewards_half(state)
            old_q_value = q_values_half[old_state_index][action_index]
            
            temporal_difference = reward + (discount_factor * np.max(q_values[state_index])) - old_q_value
        
            #update the Q-value for the previous state and action pair
            new_q_value = old_q_value + (learning_rate * temporal_difference)
            q_values_half[old_state_index][action_index] = new_q_value
            trial_num = trial_num + 1
        
    print('Training complete!')


def is_half_terminal_state(state):
    
    if state[0][n_half-1]>0 and state[n_half-1][n-1]>0:
        return True
    else:
        return False
    
def is_connected_state(current_state):
    flag=0
    for k in range(n_half-1):
        for l in range(n_half,n):
            if current_state[k][l]>0:
                flag=1
    if flag==1:
        return(True)
    else:
        return(False)
    
def training_network_bootstrap(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes):
    #run through 500 training episodes
    for episode in range(int(total_episodes)):
        if episode%1000==0:
            print(episode)
#             if episode>150000 and epsilon<0.9:
#                 epsilon+=0.05
        
        #get the starting state for this episode
        state = get_starting_state()
#         print(state)
        
        #continue taking actions until we reach a terminal state or we do 1000 actions
        trial_num = 0
        while trial_num<200 and is_terminal_state(state)==False:
            
            
            #choose which action to take
            if is_connected_state(state) or is_half_terminal_state(state):
                action = get_next_action(state, epsilon)
            else:
                action = get_next_action(state, 1.)
            
            #perform the chosen action, and transition to the next state
            old_state = np.zeros([n,n]) #store the old state
            for i in range(n):
                for j in range(n):
                    old_state[i][j] = state[i][j]
                    
            state = get_next_state(old_state, action, p_l, p_bm)
            
            old_state_index = get_state_index(old_state)
            state_index = get_state_index(state)
            action_index = get_action_index(action)
            
            #receive the reward for moving to the new state, and calculate the temporal difference
            reward = rewards(state_index)
            old_q_value = q_values[old_state_index][action_index]
            
            if is_connected_state(old_state) or is_half_terminal_state(old_state):
                temporal_difference = reward + (discount_factor * np.max(q_values[state_index])) - old_q_value
            else:
                temporal_difference = 0

            #update the Q-value for the previous state and action pair
            new_q_value = old_q_value + (learning_rate * temporal_difference)
            q_values[old_state_index][action_index] = new_q_value
            trial_num = trial_num + 1
       
        if episode%10000==0:
            np.save("q_valuesn7m3_pbm0.75_wt_bootstrap_trained50000"+str(p_l)+".npy",q_values)
            
    print('Training complete!')

In [7]:
#Define a function that will get the shortest path between any initial state and final state.
        
def evolve_state(start_state, p_l, p_bm, steps, printing=False):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        #get the best action to take
        action = get_next_action(current_state, 1.)
        if printing:
            print(np.array(action))
            print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
        if printing:
            print(np.array(current_state))
            print("")
        evolution_path.append(current_state)
        trial_num = trial_num + 1
        
    return evolution_path

# #Define a function that will get the shortest path between any initial state and final state.
# def evolve_action(start_state, p_l, p_bm, steps,n_node,m_star,cta):
#     trial_num = 0
#     #if this is a 'legal' starting state
#     current_state = np.array([i for i in start_state])
#     evolution_path = []
#     #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
#     while trial_num<steps:
#         #get the best action to take
#         action = get_next_action(current_state, 1.,n_node,m_star,cta)
#         #move to the next location on the path, and add the new location to the list
#         current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm,n_node,m_star)])
#         evolution_path.append(action)
#         trial_num = trial_num + 1
        
#     return evolution_path

In [16]:
def shortest_path(start_state, p_l, p_bm,n_node,m_star,cta, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state,n_node,m_star):
        #get the best action to take
        action = get_next_action(current_state, 1.,n_node,m_star,cta)
        
        flag=0
        for i in range(n_node-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n_node-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm,n_node,m_star)])
#         print(current_state)
#         print("")
        if cc==False:
            if flag==1:
                evolution_path.append(current_state)
        else:
            evolution_path.append(current_state)
        trial_num+=1
    evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path
    else:
        return evolution_path,nr

In [9]:
def shortest_path(start_state, p_l, p_bm, steps, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state):
        #get the best action to take
        action = get_next_action(current_state, 1.)
        flag=0
        
        for i in range(n-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
#         print(current_state)
#         print("")
        if cc==False:
            if flag==1:
                evolution_path.append(current_state)
        else:
            evolution_path.append(current_state)
        trial_num+=1
    evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path
    else:
        return evolution_path,nr

In [18]:
total_episodes = 50000
learning_rate = 0.01
for p_l in [0.5]:  
    p_bm = 0.75
    epsilon = 0.25
    discount_factor = 0.8

    t0=time.time()
    training_network_half(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes)
    print(time.time()-t0)
    np.save("q_valuesn4m3_pbm0.75_wt_pl"+str(p_l)+".npy", q_values)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
237.84365677833557
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
108.62693500518799
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
63.73627591133118
0
1000
2000
3000
4000
5000

In [10]:
#combining policies from two chains --- bootstrapping

m_star = 3  #number of states = m_star + 1
n_node = 7  #number of nodes

q_values = np.zeros([n_state,n_action])
all_states_c = []

def is_connected_state(current_state):
    flag=0
    for k in range(n_half-1):
        for l in range(n_half,n):
            if current_state[k][l]>0:
                flag=1
                break
    if flag==1:
        return(True)
    else:
        return(False)
    

for i in range(n_state):
    if is_connected_state(all_states[i]):
        all_states_c.append(all_states[i])

all_states_c = np.array(all_states_c)
print(len(all_states_c))

10071


In [11]:
def get_random_state_c():
    i = np.random.randint(0,len(all_states_c))
    return all_states_c[i]

def get_starting_state_n():
    
#   get a random state
    current_state = get_random_state_c()
    
#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state(current_state):
        current_state = get_random_state_c()

    return current_state

In [12]:
for p_l in [0.5]:
    q_values = np.zeros([n_state,n_action])
    q_values_half_chain = np.load("q_valuesn4m3_pbm0.75_wt_pl"+str(0.5)+".npy",allow_pickle=True)
    c_node = int((n-1)/2)
    
    for i in range(len(all_states)):
        print(i)
        for j in range(len(all_actions)):
            if not is_connected_state(all_states[i]) and all_actions[j][c_node][c_node]==0:
                i1 = get_state_index_half(all_states[i][0:c_node+1,0:c_node+1])
                j1 = get_action_index_half(all_actions[j][0:c_node+1,0:c_node+1])
                i2 = get_state_index_half(all_states[i][c_node:n,c_node:n])
                j2 = get_action_index_half(all_actions[j][c_node:n,c_node:n]) 
            
                q_values[i][j] = 0.5*(q_values_half_chain[i1][j1] + q_values_half_chain[i2][j2])
    np.save("q_valuesn7m3_pbm0.75_wt_bootstrap_guess"+str(p_l)+".npy",q_values)

FileNotFoundError: [Errno 2] No such file or directory: 'q_valuesn4m3_pbm0.75_wt_pl0.5.npy'

In [23]:
total_episodes = 50000
learning_rate = 0.01
for p_l in [0.3,0.5,0.7]:  
    p_bm = 0.5
    epsilon = 0.5
    discount_factor = 0.8

    q_values = np.load("q_valuesn5m3_pbm0.5_wt_bootstrap_guess"+str(p_l)+".npy",allow_pickle=True)
    q_values = q_values[()].A
    # q_values = np.zeros([n_state,n_action])

    t0=time.time()
    training_network_n(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes,n_node,m_star,ct,cta)
    print(time.time()-t0)
    np.save("q_valuesn5m3_pbm0.5_wt_bootstrap_trained50000"+str(p_l)+".npy",csc_matrix(q_values))

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
997.0370328426361
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
624.778529882431
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
Training complete!
546.2679197788239


In [146]:
q_values = np.load("q_valuesn5m3_pl0.5_pbm0.5_wt_bootstrap_trained100000.npy",allow_pickle=True)
q_values = q_values[()].A
p_l = 0.5
p_bm = 0.5
start_state = np.zeros([5,5])
print(start_state)
print("")
evolve_state(start_state, p_l, p_bm, 20, n_node, m_star, cta, printing=True)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 1. 0. 0. 0.]
 [1. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

[[0. 1. 0. 0. 0.]
 [1. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

[[0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

[[0. 0. 2. 0. 0.]
 [0. 0. 0. 0. 0.]
 [2. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]]

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 0. 3. 0. 0.]
 [0. 0. 0. 0. 0.]
 [3. 0. 0. 1. 0.]
 [0. 0. 1. 0. 2.]
 [0. 0. 0. 2. 0.]]

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 0. 3. 0. 0.]
 [0. 0. 0. 0. 0.]
 [3. 0. 0. 0. 2.]
 [0. 0. 0. 0. 0.]
 [0. 0. 

[array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]),
 array([[0., 1., 0., 0., 0.],
        [1., 0., 1., 0., 0.],
        [0., 1., 0., 1., 0.],
        [0., 0., 1., 0., 1.],
        [0., 0., 0., 1., 0.]]),
 array([[0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]),
 array([[0., 0., 2., 0., 0.],
        [0., 0., 0., 0., 0.],
        [2., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.]]),
 array([[0., 0., 3., 0., 0.],
        [0., 0., 0., 0., 0.],
        [3., 0., 0., 1., 0.],
        [0., 0., 1., 0., 2.],
        [0., 0., 0., 2., 0.]]),
 array([[0., 0., 3., 0., 0.],
        [0., 0., 0., 0., 0.],
        [3., 0., 0., 0., 2.],
        [0., 0., 0., 0., 0.],
        [0., 0., 2., 0., 0.]]),
 array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]

In [91]:
q_values_half_chain[1]

array([30.78899348,  0.        , 28.55754902,  0.        , 26.22564933,
        0.        ,  0.        ,  0.        , 39.89025056,  0.        ,
       34.27944591,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

In [27]:
#average waiting time K_n without CC
n_node = 4
m_star = 3
mean_list=[]
mean_list_fid=[]

for p_l in np.linspace(0.3,0.9,7):
    p_bm = 0.5
    
    q_values = np.load("q_valuesn4m3_400000_pbm0.5_wtp_l0.4.npy",allow_pickle=True)
    q_values = q_values[()].A

    mean_mean=[]
    mean_fid=[]
    for p in range(10):
        print(p)
        K_n = []
        K_n_fid = []
        for k in range(1000):
    #         print(k)
            state = np.zeros([n_node, n_node])
            K_n.append(len(shortest_path(state, p_l, p_bm, n_node, m_star, cta, cc=False, nreq=False))-2)
            K_n_fid.append(shortest_path(state, p_l, p_bm, n_node, m_star, cta, cc=False, nreq=False)[-1][0][n_node-1])

        mean_mean.append(np.mean(K_n))
        mean_fid.append(np.mean(K_n_fid))

    print(np.mean(mean_mean),np.std(mean_mean))
    print(np.mean(mean_fid))
    mean_list.append(np.mean(mean_mean))
    mean_list_fid.append(np.mean(mean_fid))

0
1
2
3
4
5
6
7
8
9
34.429500000000004 1.08194780373177
2.3699
0
1
2
3
4
5
6
7
8
9
20.2465 0.5366095880619354
2.2750999999999997
0
1
2
3
4
5
6
7
8
9
13.6103 0.42040291388143364
2.167
0
1
2
3
4
5
6
7
8
9
10.282699999999998 0.31783299073570076
2.039
0
1
2
3
4
5
6
7
8
9
7.890899999999999 0.1722175658868747
1.8582
0
1
2
3
4
5
6
7
8
9
6.3588000000000005 0.1857997847146223
1.6405999999999998
0
1
2
3
4
5
6
7
8
9
5.1989 0.13428585182363798
1.3508


In [28]:
mean_list

[34.429500000000004,
 20.2465,
 13.6103,
 10.282699999999998,
 7.890899999999999,
 6.3588000000000005,
 5.1989]